<img src='https://www.icos-cp.eu/sites/default/files/2017-11/ICOS_CP_logo.png' width=400 align=right>

# ICOS Carbon Portal Python Libraries

This example uses a foundational library called `icoscp_core` which can be used to access time-series ICOS data that are *previewable* in the ICOS Data Portal. "Previewable" means that it is possible to visualize the data variables in the preview plot. The library can also be used to access (meta-)data from [ICOS Cities](https://citydata.icos-cp.eu/portal/) and [SITES](https://data.fieldsites.se/portal/) data repositories.

General information on all ICOS Carbon Portal Python libraries can be found on our [help pages](https://icos-carbon-portal.github.io/pylib/).

Documentation of the `icoscp_core` library, including information on running it locally, can also be found on [PyPI.org](https://pypi.org/project/icoscp_core/).

Note that for running this example locally, authentication is required (see the `how_to_authenticate.ipynb` notebook).

# Example: access and work with atmospheric Near Real Time data (Level 1)

> **All the settings you need are in the Configuration cell below.** Station, gas and country are
> defined once and reused everywhere: the data-object URIs, the filename pattern, the plotted column
> and the plot labels are all derived from them. You do not need to edit the same value in several
> cells or copy filenames by hand.

**Folder management: to avoid accumulating files across repeated runs, this notebook empties its `outputs/` and `exports/` folders at the start of each run. Use the export cell at the end to keep anything you want before running it again.**

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* in two situations:
>
> * **before running the notebook again**, so that the interactive widgets are rebuilt from scratch.
>   Widget state belongs to the kernel that created it, so leftovers from a previous run can make
>   controls stop responding or disappear;
> * **before saving**, so that the file stays small and does not carry stale widget state. Plotly
>   embeds a large JavaScript bundle into saved outputs, and reopening a notebook that stores widget
>   state shows "Error displaying widget: model not found".
>
> If controls are still unresponsive after clearing, restart the kernel (*Kernel → Restart Kernel*)
> and, should that not be enough, reload the browser page: the widgets live partly in the page
> itself, and a reload is sometimes the only way to clear what is left there.

## Before you start: work in your home folder

This notebook lives in the **shared VRE workspace**, under `/workspace/VREFolders/Wildfires_in-Europe/`.
That location is convenient for distributing tools, but it is a FUSE-mounted shared filesystem:
working there is noticeably slower, and only the owner of a folder can write to it — everyone else
sees it read-only.

Your home directory (`/home/jovyan`, shown as `/` at the top of the JupyterLab file browser) is a
persistent volume of your own, and it is where you should run things from.
**Copy the tool into your home directory and work on that copy.**

Open a terminal (*File → New → Terminal*) and run:

```bash
cp -r /workspace/VREFolders/Wildfires_in-Europe/icos_vre ~/icos_vre
cd ~/icos_vre
```

Then open the copied notebook from the file browser and use that copy.

> **Copy the whole folder, not just the `.ipynb` file.** This notebook is self-contained, so the `.ipynb` alone would work - but copying the folder keeps it together with the other ICOS examples and with anything it writes.

A few practical notes:

* Your home directory is **persistent** across sessions, and private to you. When you want to share
  your work with the other VRE users, copy the folder back into the shared workspace.
* It is network storage (NFS), not a local disk, and the space is shared with the other users. Data
  that this notebook downloads can grow quickly, so it is worth deleting what you no longer need.
* On NFS, deleting a folder whose files are still open fails with
  `Device or resource busy: '.nfs0000...'`. If that happens, shut down the kernel that was using
  those files (*Kernel → Shut Down Kernel*, or the "Running Terminals and Kernels" panel) and try
  again.


## 0. Configuration

* `SELECTED_HEIGHT` — measurement height in metres; `None` picks the first height available
* `MAX_OBJECTS_TO_COMBINE` — safety limit for the final section, which downloads one file per day
* `DEFAULT_*` — used only if you skip one of the selectors below

**Station** (section 2) and **data type** (section 4) are chosen from dropdowns. Everything else —
URIs, filename pattern, plotted column, axis labels — is derived from those two choices.

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

# Identity of this notebook. The file itself carries no version number - only one version is
# published at a time - so the version is printed here instead, which is also what tells you
# whether the kernel is running the file you think it is.
NOTEBOOK_NAME = "ICOS atmospheric NRT"
NOTEBOOK_VERSION = "2.4"
NOTEBOOK_DATE = "2026-09-15"

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

SELECTED_HEIGHT = None     # metres; None = first available height
MAX_OBJECTS_TO_COMBINE = 200

# Fallbacks used if you skip a selector below (or forget to press its Confirm button)
DEFAULT_COUNTRY = "IT"
DEFAULT_STATION_CODE = "CMN"
DEFAULT_GAS = "CO"

# Map rendering backend for the station map in section 2:
#   "plotly"     interactive map with a geographic background (recommended)
#   "matplotlib" static scatter, no basemap; use it if the interactive one stays blank
MAP_BACKEND = "plotly"

# Column names shared by the ATC NRT products. The quality-flag column is resolved per
# variable in section 7: gas products carry a single 'Flag' column, while the meteo product
# has one flag per variable ('AT-Flag', 'RH-Flag', ...).
TIME_COLUMN = "TIMESTAMP"
DEFAULT_QUALITY_FLAG = "Flag"
VALUE_ACCEPT_QUALITY = "U"

NOTEBOOK_SLUG = "icos_NRT"


## 1. Import libraries

In [ ]:
import os
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
    "icoscp_core": "icoscp",
    # Hidden dependency: plotly needs nbformat to render figures, but does not declare it.
    "nbformat": "nbformat",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                   check=False, cwd=os.path.expanduser("~"))
else:
    print("All required packages are already installed.")

import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode

# Loads the Plotly JavaScript into the page. Without it, Plotly figures shown inside an
# ipywidgets Output (the station map in section 2) render as an empty area. connected=True
# pulls the library from the CDN instead of embedding ~5 MB into the saved notebook.
if MAP_BACKEND == "plotly":
    init_notebook_mode(connected=True)
import ipywidgets as widgets
from IPython.display import display

from icoscp_core.icos import data, meta, ATMO_STATION


# --- Outputs of this run -------------------------------------------------------------
# Figures are shown inline and kept in memory; nothing is written to disk while you work.
# The export cell at the end of the notebook writes out what you decide to keep, into the
# folder below, and packs it into a single zip you can download.
#
# The folder is emptied at the start of each run, so that repeated runs do not accumulate
# files. Set CLEAN_OUTPUTS to False to keep what earlier runs produced.
import shutil
from pathlib import Path

OUTPUT_DIR = Path("outputs")
CLEAN_OUTPUTS = True

if CLEAN_OUTPUTS and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
        print("Note: some files could not be removed (still open elsewhere); continuing anyway.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRAWN_FIGURES = {}


def register_figure(fig, stem):
    """Remember a figure under a name, for the export cell at the end."""
    DRAWN_FIGURES[stem] = fig

# Archives produced by the export cell at the end go here. The folder is emptied together with
# the rest, so the zips of previous runs do not pile up next to the notebook.
EXPORT_DIR = Path("exports")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Select the station

Pick a **country** first (or *All countries*), then a **station** from that country, and press
**Confirm station**. The map shows the stations of the selected country, with the chosen one in red.

⚠️ Selecting a value in the dropdown is not enough — you must press the button, otherwise the
notebook falls back to `DEFAULT_STATION_CODE` and tells you so. This cell is safe to re-run: an
already-confirmed station is preserved.

In [ ]:
stations = meta.list_stations(ATMO_STATION)
print(f"{len(stations)} ATMO stations available.")

stations_by_code = {s.id: s for s in stations}

def _country_options():
    codes = sorted({s.country_code for s in stations if s.country_code})
    return [("All countries", "ALL")] + [(c, c) for c in codes]

def _stations_for(country):
    sel = [s for s in stations if country == "ALL" or s.country_code == country]
    return sorted((f"{s.id} - {s.name} ({s.country_code})", s.id) for s in sel)

# Preserve an already-confirmed station if this cell is re-run
if "SELECTED_STATION" not in globals():
    SELECTED_STATION = None
_initial_code = SELECTED_STATION if SELECTED_STATION in stations_by_code else DEFAULT_STATION_CODE
if _initial_code not in stations_by_code:
    _initial_code = stations[0].id
_initial_country = stations_by_code[_initial_code].country_code or "ALL"

country_dd = widgets.Dropdown(options=_country_options(), value=_initial_country,
                              description="Country:", style={"description_width": "initial"},
                              layout=widgets.Layout(width="320px"))
station_dd = widgets.Dropdown(options=_stations_for(_initial_country), value=_initial_code,
                              description="Station:", style={"description_width": "initial"},
                              layout=widgets.Layout(width="620px"))
confirm_btn = widgets.Button(description="Confirm station", button_style="success", icon="check")
confirm_out = widgets.Output()
map_out = widgets.Output()

# Set while the dropdowns are repopulated programmatically, so the map is redrawn exactly once
# per user action instead of once per cascading update.
_updating = {"busy": False}

GEO_LAYOUT = dict(
    resolution=50, showcountries=True, countrycolor="#888",
    showland=True, landcolor="#f2efe9", showocean=True, oceancolor="#dbe9f2",
    showlakes=False, showcoastlines=True, coastlinecolor="#999",
)

map_out = widgets.Output()
map_area = map_out

def _draw_map_plotly(located, selected, current):
    others = [s for s in located if s.id != current]
    figure = go.Figure()
    if others:
        figure.add_trace(go.Scattergeo(
            lon=[s.lon for s in others], lat=[s.lat for s in others],
            text=[f"{s.id} - {s.name}" for s in others],
            hoverinfo="text", mode="markers", name="Stations",
            marker=dict(size=8, color="#3B4872", line=dict(width=1, color="white"))))
    if selected:
        figure.add_trace(go.Scattergeo(
            lon=[s.lon for s in selected], lat=[s.lat for s in selected],
            text=[f"{s.id} - {s.name}" for s in selected],
            hoverinfo="text", mode="markers", name="Selected",
            marker=dict(size=16, color="crimson", symbol="star",
                        line=dict(width=1.5, color="white"))))
    figure.update_geos(fitbounds="locations", **GEO_LAYOUT)
    figure.update_layout(height=420, margin=dict(l=0, r=0, t=36, b=0), showlegend=False,
                         title=dict(text=f"{len(located)} ATMO station(s) - selected: {current}",
                                    font=dict(size=13)))
    figure.show()

def _draw_map_static(located, selected, current):
    """Static alternative: a plain scatter, no basemap. Selected with MAP_BACKEND."""
    if True:
        others = [s for s in located if s.id != current]
        lons = [s.lon for s in located]
        lats = [s.lat for s in located]

        fig, ax = plt.subplots(figsize=(7, 4.5))
        if others:
            ax.scatter([s.lon for s in others], [s.lat for s in others],
                       s=45, c="#3B4872", edgecolors="white", zorder=3, label="Stations")
        if selected:
            ax.scatter([s.lon for s in selected], [s.lat for s in selected],
                       s=160, c="crimson", edgecolors="white", zorder=4, marker="*",
                       label="Selected")

        # A single far-away station (e.g. La Reunion for FR) would squash all the others into
        # an unreadable cluster, so the view is framed on the bulk of the stations.
        def _bulk_range(values):
            """Range of the values, discarding the extremes when there are enough points."""
            ordered = sorted(values)
            if len(ordered) < 5:
                return ordered[0], ordered[-1]
            k = max(1, len(ordered) // 10)      # number of points trimmed at each end
            return ordered[k], ordered[-(k + 1)]

        lon_lo, lon_hi = _bulk_range(lons)
        lat_lo, lat_hi = _bulk_range(lats)
        for s in selected:                      # always keep the selected station in view
            lon_lo, lon_hi = min(lon_lo, s.lon), max(lon_hi, s.lon)
            lat_lo, lat_hi = min(lat_lo, s.lat), max(lat_hi, s.lat)
        pad = max(1.5, 0.15 * max(lon_hi - lon_lo, lat_hi - lat_lo))
        ax.set_xlim(lon_lo - pad, lon_hi + pad)
        ax.set_ylim(lat_lo - pad, lat_hi + pad)

        visible = [s for s in located
                   if lon_lo - pad <= s.lon <= lon_hi + pad and lat_lo - pad <= s.lat <= lat_hi + pad]
        for s in (visible if len(visible) <= 12 else selected):
            ax.annotate(s.id, (s.lon, s.lat), textcoords="offset points", xytext=(6, 4),
                        fontsize=8, zorder=5)

        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        title = f"{len(located)} ATMO station(s) - selected: {current}"
        if len(visible) < len(located):
            title += f" ({len(located) - len(visible)} outside the view)"
        ax.set_title(title, fontsize=11)
        ax.grid(True, linestyle=":", alpha=0.6)
        mean_lat = sum(lats) / len(lats)
        ax.set_aspect(1.0 / max(0.2, math.cos(math.radians(mean_lat))), adjustable="box")
        if others and selected:
            ax.legend(loc="best", fontsize=8)
        plt.tight_layout()
        plt.show()

def _draw_map(*_):
    shown = [stations_by_code[c] for _, c in station_dd.options]
    located = [s for s in shown if s.lat is not None and s.lon is not None]
    current = station_dd.value
    selected = [s for s in located if s.id == current]
    with map_out:
        map_out.clear_output(wait=True)
        if not located:
            print("No coordinates available for these stations.")
            return
        if MAP_BACKEND == "plotly":
            _draw_map_plotly(located, selected, current)
        else:
            _draw_map_static(located, selected, current)

def _on_country_change(change):
    opts = _stations_for(change["new"])
    _updating["busy"] = True
    try:
        station_dd.options = opts
        if opts:
            station_dd.value = opts[0][1]
    finally:
        _updating["busy"] = False
    _draw_map()

def _on_station_change(change):
    if not _updating["busy"]:
        _draw_map()

def _on_confirm(_):
    global SELECTED_STATION
    SELECTED_STATION = station_dd.value
    s = stations_by_code[SELECTED_STATION]
    with confirm_out:
        confirm_out.clear_output()
        print(f"Confirmed station: {s.id} - {s.name} ({s.country_code})")

country_dd.observe(_on_country_change, names="value")
station_dd.observe(_on_station_change, names="value")
confirm_btn.on_click(_on_confirm)

display(widgets.VBox([country_dd, station_dd, map_area, confirm_btn, confirm_out]))
_draw_map()

with confirm_out:
    if SELECTED_STATION:
        print(f"Confirmed station: {SELECTED_STATION} (already confirmed - you can move on)")
    else:
        print(f"Not confirmed yet - will default to {DEFAULT_STATION_CODE}.")

### Apply the selection

Everything the rest of the notebook needs is derived here from the confirmed station and from `GAS`.

In [ ]:
SELECTED_STATION = globals().get("SELECTED_STATION")
if SELECTED_STATION not in stations_by_code:
    SELECTED_STATION = DEFAULT_STATION_CODE
    print(f"No station confirmed - defaulting to {SELECTED_STATION}.")

station_obj = stations_by_code[SELECTED_STATION]
STATION_CODE = station_obj.id

# The station's own URI is used rather than a reconstructed string. The metadata service uses
# https for station metadata but http in data-object queries, so both spellings are derived here.
_uri = station_obj.uri
STATION_URI_HTTPS = _uri.replace("http://", "https://", 1)
STATION_URI_HTTP = _uri.replace("https://", "http://", 1)

print(f"Station: {STATION_CODE} - {station_obj.name} ({station_obj.country_code})")

## 3. View metadata for the selected station

Some of the metadata associated with the station configured above.

In [ ]:
station_meta = meta.get_station_meta(STATION_URI_HTTPS)

print("Name:", station_meta.org.name)
print("Got ICOS label on:", station_meta.specificInfo.labelingDate)

print("Known staff, possibly former:")
pd.DataFrame([
    {
        "first_name": memb.person.firstName,
        "last_name": memb.person.lastName,
        "role": memb.role.role.label,
        "start": memb.role.start,
        "end": memb.role.end,
    }
    for memb in station_meta.staff
])

## 4. Select the data type

Data types are the most important category of classification of data objects. They exist to combine a number of other metadata elements, so that data objects are associated with a data type instead of being linked to numerous separate metadata fields.

The dropdown lists only previewable **Level 1 (Near Real Time)** data types of the atmospheric domain. See more information [about data levels](https://www.icos-cp.eu/data-services/data-collection/data-levels-quality).

Pick one and press **Confirm data type**. The measured species and the name of the data column are derived from the selected type, so nothing else needs to be edited.

⚠️ As with the station selector, choosing a value is not enough — press the button, otherwise the notebook falls back to `DEFAULT_GAS` and says so.

In [ ]:
all_data_types = meta.list_datatypes()

selected_datatypes = [
    dt for dt in all_data_types
    if dt.has_data_access
    and dt.data_level == 1
    and dt.theme.uri == "http://meta.icos-cp.eu/resources/themes/atmosphere"
]
print(f"{len(selected_datatypes)} previewable Level 1 atmospheric data type(s).")

datatypes_by_uri = {dt.uri: dt for dt in selected_datatypes}

def _gas_from_uri(uri):
    """Derive the species code from a data-type URI.

    Handles the plain NRT products (.../atcCh4NrtDataObject -> CH4) as well as the
    "growing time series" and FastTrack variants, whose URIs carry extra tokens
    (.../atcCh4NrtGrowingDataObject -> CH4).
    """
    m = re.search(r"/atc(.+?)DataObject$", uri)
    if not m:
        return None
    token = m.group(1)
    for kind in ("FastTrack", "Growing", "Nrt", "L1", "L2", "Generic"):
        token = token.replace(kind, "")
    return token.upper() or None

def _dt_label(dt):
    """Dropdown label: species code when it can be derived, otherwise the URI's short name,
    so that an entry is never shown as an anonymous '?'."""
    code = _gas_from_uri(dt.uri) or dt.uri.rsplit("/", 1)[-1]
    return f"{code} - {dt.label}"

_dt_options = sorted((_dt_label(dt), dt.uri) for dt in selected_datatypes)

# Preserve an already-confirmed choice if this cell is re-run
if "SELECTED_DATA_TYPE" not in globals():
    SELECTED_DATA_TYPE = None
# Match on the species code derived from the URI rather than rebuilding the URI by hand:
# capitalising "CO" gives "Co", and one product spelled differently would silently send the
# notebook to whatever comes first in the list. Plain NRT products are preferred over the
# Growing and FastTrack variants of the same species.
def _default_for(gas):
    wanted = gas.upper()
    matches = [uri for uri in datatypes_by_uri if _gas_from_uri(uri) == wanted]
    if not matches:
        return None
    plain = [u for u in matches if "Growing" not in u and "FastTrack" not in u]
    return (plain or matches)[0]


_default_uri = _default_for(DEFAULT_GAS)
_initial_uri = SELECTED_DATA_TYPE if SELECTED_DATA_TYPE in datatypes_by_uri else _default_uri
if _initial_uri not in datatypes_by_uri:
    print(f"No NRT product found for {DEFAULT_GAS} - falling back to the first in the list.")
    _initial_uri = _dt_options[0][1]

datatype_dd = widgets.Dropdown(options=_dt_options, value=_initial_uri,
                               description="Data type:", style={"description_width": "initial"},
                               layout=widgets.Layout(width="760px"))
confirm_dt_btn = widgets.Button(description="Confirm data type", button_style="success", icon="check")
confirm_dt_out = widgets.Output()

def _on_confirm_dt(_):
    global SELECTED_DATA_TYPE
    SELECTED_DATA_TYPE = datatype_dd.value
    with confirm_dt_out:
        confirm_dt_out.clear_output()
        print(f"Confirmed: {_gas_from_uri(SELECTED_DATA_TYPE)} - "
              f"{datatypes_by_uri[SELECTED_DATA_TYPE].label}")

confirm_dt_btn.on_click(_on_confirm_dt)
display(widgets.VBox([datatype_dd, confirm_dt_btn, confirm_dt_out]))

with confirm_dt_out:
    if SELECTED_DATA_TYPE:
        print(f"Confirmed: {_gas_from_uri(SELECTED_DATA_TYPE)} (already confirmed - you can move on)")
    else:
        print(f"Not confirmed yet - will default to {DEFAULT_GAS}.")

### Apply the selection

The species code and the data column are derived from the data type chosen above.

In [ ]:
SELECTED_DATA_TYPE = globals().get("SELECTED_DATA_TYPE")
if SELECTED_DATA_TYPE not in datatypes_by_uri:
    SELECTED_DATA_TYPE = _default_uri
    print(f"No data type confirmed - defaulting to {DEFAULT_GAS}.")
    if SELECTED_DATA_TYPE not in datatypes_by_uri:
        raise RuntimeError(
            f"The default data type is not available either:\n  {SELECTED_DATA_TYPE}\n"
            f"Pick one from the dropdown above and press Confirm."
        )

DATA_TYPE = SELECTED_DATA_TYPE
GAS = _gas_from_uri(DATA_TYPE) or DEFAULT_GAS
DATA_COLUMN = GAS.lower()   # verified against the actual columns once the data is loaded

print(f"Data type: {datatypes_by_uri[DATA_TYPE].label}")
print(f"           {DATA_TYPE}")
print(f"Species:   {GAS}  ->  expected data column '{DATA_COLUMN}'")

## 5. Find data objects for the selected station and data type

The filename pattern is built from `STATION_CODE` and `GAS`, so it stays in step with the Configuration cell automatically.

In [ ]:
station_data_objects = meta.list_data_objects(datatype=DATA_TYPE, station=STATION_URI_HTTP)

print(f"{len(station_data_objects)} data object(s) found for {STATION_CODE} / {GAS}.")

if not station_data_objects:
    # Not every station provides every product, so this is a normal outcome rather than an error
    # in the notebook. Report where the data actually is, to make the next try easy.
    print(f"\n{STATION_CODE} has no data of type '{datatypes_by_uri[DATA_TYPE].label}'.")
    print("Looking for stations that do have it ...")

    # Only the network call is guarded: a failure in the hints below must not be reported
    # as if the lookup itself had failed.
    elsewhere, lookup_error = [], None
    try:
        elsewhere = meta.list_data_objects(datatype=DATA_TYPE)
    except Exception as exc:  # noqa: BLE001 - the diagnostic must never mask the real problem
        lookup_error = f"{type(exc).__name__}: {exc}"

    hint = ""   # repeated inside the exception, so it survives a collapsed output area

    if lookup_error:
        print(f"  (could not check other stations: {lookup_error})")
        hint = f"could not check other stations ({lookup_error})"
    elif elsewhere:
        station_pattern = re.compile(r"^ICOS_ATC_(?:NRT_)?([A-Z0-9]{3})_")
        codes = sorted({m.group(1) for o in elsewhere
                        if (m := station_pattern.match(o.filename))})
        if codes:
            named = [f"{c} ({stations_by_code[c].name})" if c in stations_by_code else c
                     for c in codes]
            print(f"  available at {len(codes)} station(s): {', '.join(named)}")
            hint = (f"this data type exists at {len(codes)} other station(s) "
                    f"({', '.join(codes)}) - pick one of them in section 2")
        else:
            print(f"  {len(elsewhere)} object(s) exist for this data type, but the station "
                  f"could not be read from their filenames.")
            hint = "this data type has data, but at stations that could not be identified"
    else:
        # Registered in the catalogue but holding no objects at all: typically a product that
        # was superseded. Point at the sibling types of the same species.
        print("  no data objects at all for this data type - it appears to be empty, "
              "possibly superseded by another product.")
        hint = "this data type is empty across the whole archive"
        siblings = sorted(
            d.label for u, d in datatypes_by_uri.items()
            if u != DATA_TYPE and _gas_from_uri(u) == GAS
        )
        if siblings:
            print(f"  other data type(s) for {GAS} in the dropdown:")
            for lbl in siblings:
                print(f"    - {lbl}")
            print("  Section 5 reads both the daily and the growing filename conventions, "
                  "so any of these that holds data will work.")
            hint += (f"; try another {GAS} data type in section 4: "
                     f"{', '.join(siblings)}")

    raise RuntimeError(
        f"No data objects for {STATION_CODE} / {GAS}. " +
        (f"Reason: {hint}." if hint else
         "Pick another station in section 2, or another data type in section 4.")
    )

# Two filename conventions are in use:
#   daily NRT       ICOS_ATC_NRT_<stn>_<date>_<same date>_<height>_<id>_<species>.zip
#   growing series  ICOS_ATC_NRT_<stn>_<start>_<end>_<height>_<id>_<species>.zip   (cumulative)
# A daily file is simply the case where start == end.
#
# The species token is captured rather than required to equal GAS: the two do not always agree
# (the Meteo data type yields GAS='METEO' while its files are named 'MTO'), and matching it is
# redundant anyway, since the query above already restricted the objects to this data type.
#
# The field between the height and the species is an instrument identifier. It is usually a
# plain number, but it can also be a range such as "1690-1793" when a file spans an instrument
# change, so it is matched loosely rather than as digits only.
pattern = re.compile(
    rf"^ICOS_ATC_(?:NRT_)?{STATION_CODE}_"
    rf"(\d{{4}}-\d{{2}}-\d{{2}})_(\d{{4}}-\d{{2}}-\d{{2}})_"
    rf"(\d+(?:\.\d+)?)_[\w.-]+_([A-Za-z0-9]+)\.zip$"
)

file_groups = {}
object_dates = {}       # filename -> (start, end) parsed from the filename
file_species = set()    # species tokens actually seen in the filenames
unmatched = []
for obj in station_data_objects:
    m = pattern.match(obj.filename)
    if m:
        file_groups.setdefault(float(m.group(3)), []).append(obj)
        object_dates[obj.filename] = (m.group(1), m.group(2))
        file_species.add(m.group(4))
    else:
        unmatched.append(obj.filename)

def date_label(filename):
    """'2026-07-10' for a daily file, '2026-04-01 to 2026-08-14' for a growing one."""
    start, end = object_dates[filename]
    return start if start == end else f"{start} to {end}"

print("Available measurement heights:", sorted(file_groups.keys()))
if object_dates:
    starts = sorted(s for s, _ in object_dates.values())
    ends = sorted(e for _, e in object_dates.values())
    print(f"Period covered: {starts[0]} to {ends[-1]} ({len(object_dates)} file(s))")
if file_species and file_species != {GAS}:
    print(f"Species token in the filenames: {', '.join(sorted(file_species))} "
          f"(the data type is named '{GAS}')")
if unmatched:
    print(f"\n{len(unmatched)} filename(s) did not match the expected pattern and were ignored:")
    for name in unmatched[:5]:
        print(f"  {name}")
    if len(unmatched) > 5:
        print(f"  ... and {len(unmatched) - 5} more")
if not file_groups:
    raise RuntimeError(
        "No filename matched the expected pattern. The filenames listed above use a convention "
        "this notebook does not know; the station and data type themselves are fine."
    )

## 6. Choose height and dates

Each NRT file covers **one day at one measurement height**, so plotting a longer period means
selecting several files.

Pick the height first — the list of dates follows it — then the dates: ctrl-click to add one,
shift-click to take a range. The selected files are concatenated into a single time series. Where
two of them overlap, as the daily files do with the growing series, each instant is kept once.

Press **Confirm dates** when you are done. Without confirming, the most recent file at the selected
height is used.

In [ ]:
height = SELECTED_HEIGHT if SELECTED_HEIGHT is not None else sorted(file_groups)[0]
if height not in file_groups:
    raise RuntimeError(f"Height {height} not available. Choose among {sorted(file_groups)}.")

def _objects_for(h):
    """Data objects at a given height, most recent first."""
    return sorted(file_groups[h], key=lambda o: object_dates[o.filename], reverse=True)

# Preserve already-confirmed choices if this cell is re-run
if "SELECTED_FILES" not in globals():
    SELECTED_FILES = None

height_dd = widgets.Dropdown(options=sorted(file_groups), value=height,
                             description="Height (m):", style={"description_width": "initial"},
                             layout=widgets.Layout(width="260px"))
# NRT publishes one file per day, so a single file is a single day. Several can be selected -
# ctrl-click, or shift-click for a range - and they are concatenated into one time series.
date_dd = widgets.SelectMultiple(
    options=[(f"{date_label(o.filename)}  ({o.filename})", o.filename)
             for o in _objects_for(height)],
    rows=10, description="Dates:", style={"description_width": "initial"},
    layout=widgets.Layout(width="720px"))
if SELECTED_FILES:
    keep = tuple(f for f in SELECTED_FILES if f in [x for _, x in date_dd.options])
    if keep:
        date_dd.value = keep
elif date_dd.options:
    date_dd.value = (date_dd.options[0][1],)

confirm_file_btn = widgets.Button(description="Confirm dates", button_style="success",
                                  icon="check")
confirm_file_out = widgets.Output()
_file_updating = {"busy": False}

def _on_height_change(change):
    opts = [(f"{date_label(o.filename)}  ({o.filename})", o.filename)
            for o in _objects_for(change["new"])]
    _file_updating["busy"] = True
    try:
        date_dd.options = opts
        if opts:
            date_dd.value = (opts[0][1],)
    finally:
        _file_updating["busy"] = False

def _on_confirm_file(_):
    global SELECTED_FILES
    SELECTED_FILES = list(date_dd.value)
    with confirm_file_out:
        confirm_file_out.clear_output()
        if not SELECTED_FILES:
            print("No date selected - pick at least one.")
            return
        print(f"Confirmed: {height_dd.value} m, {len(SELECTED_FILES)} file(s)")
        for f in SELECTED_FILES[:5]:
            print(f"   {date_label(f)}")
        if len(SELECTED_FILES) > 5:
            print(f"   ... and {len(SELECTED_FILES) - 5} more")

height_dd.observe(_on_height_change, names="value")
confirm_file_btn.on_click(_on_confirm_file)
display(widgets.VBox([height_dd, date_dd, confirm_file_btn, confirm_file_out]))

with confirm_file_out:
    if SELECTED_FILES:
        print(f"{len(SELECTED_FILES)} file(s) already confirmed - you can move on.")
    else:
        print("Not confirmed yet - will default to the most recent file at the selected height.")
        print("Select several dates to plot a longer period: ctrl-click, or shift-click a range.")

### Apply the selection and load the data

In [ ]:
height = height_dd.value
_available = [f for _, f in date_dd.options]
SELECTED_FILES = [f for f in (globals().get("SELECTED_FILES") or []) if f in _available]
if not SELECTED_FILES:
    SELECTED_FILES = [date_dd.options[0][1]]
    print(f"No date confirmed - defaulting to the most recent: {date_label(SELECTED_FILES[0])}")

# Oldest first, so the concatenated series runs forward in time
objects = sorted((o for o in file_groups[height] if o.filename in SELECTED_FILES),
                 key=lambda o: object_dates[o.filename])
print(f"Height: {height} m | {len(objects)} file(s) selected")

frames = []
for obj in objects:
    # Full metadata is needed to fetch the data; a relatively expensive call (~100 ms each)
    obj_meta = meta.get_dobj_meta(obj)
    frames.append(pd.DataFrame(data.get_columns_as_arrays(obj_meta)))

# Kept for the cells below that read units and column descriptions from the metadata
dobj_meta = meta.get_dobj_meta(objects[-1])
selected_data_object = objects[-1]

df = pd.concat(frames, ignore_index=True)

# The daily files and the growing series overlap, so the same instant can arrive twice
_time_col = TIME_COLUMN if TIME_COLUMN in df.columns else None
if _time_col:
    before = len(df)
    df = (df.sort_values(_time_col)
            .drop_duplicates(subset=[_time_col], keep="first")
            .reset_index(drop=True))
    if before != len(df):
        print(f"{before - len(df)} duplicate timestamp(s) removed "
              f"(daily files and growing series overlap).")

print(f"{len(df)} rows from {date_label(objects[0].filename)} to "
      f"{date_label(objects[-1].filename)}")
display(df)

## 7. Select the data column

The column to plot is proposed automatically from the data type (`CH4` → `ch4`), but every file
contains several columns, and for non-gas products such as meteorology the names are different
altogether. Pick the one you want and press **Confirm column**.

Each entry shows the column name together with the variable name and unit taken from the object
metadata, so you do not have to guess what `AT` or `RH` stand for.

In [ ]:
# Metadata for the columns of the loaded object: used both for the dropdown labels and, later,
# for the axis labels of the plots.
ts_meta = dobj_meta.specificInfo
station_name = ts_meta.acquisition.station.org.name
column_value_types = {col.label: col.valueType for col in ts_meta.columns}

def _column_label(col):
    vt = column_value_types.get(col)
    if vt is None:
        return col
    unit = f" [{vt.unit}]" if getattr(vt, "unit", None) else ""
    return f"{col} - {vt.self.label}{unit}"

# Ancillary columns ('AT-Stdev', 'AT-NbPoints', ...) are listed after the main variables,
# and the per-variable flag columns are left out entirely.
def _is_ancillary(col):
    return col.endswith(("-Stdev", "-NbPoints", "Stdev", "NbPoints"))

plottable = [
    col for col in df.columns
    if col != TIME_COLUMN
    and not col.endswith("Flag")
    and pd.api.types.is_numeric_dtype(df[col])
]
plottable.sort(key=lambda c: (_is_ancillary(c), c))
if not plottable:
    raise RuntimeError(f"No numeric column available to plot. Columns: {list(df.columns)}")

_column_options = [(_column_label(col), col) for col in plottable]

# Default: the column derived from the data type, if the file actually has it
_proposed = DATA_COLUMN if DATA_COLUMN in plottable else None
if _proposed is None:
    _case_match = [col for col in plottable if col.lower() == DATA_COLUMN.lower()]
    _proposed = _case_match[0] if _case_match else plottable[0]
    print(f"NOTE: column '{DATA_COLUMN}' is not in this file; proposing '{_proposed}' instead.")

if "SELECTED_COLUMN" not in globals():
    SELECTED_COLUMN = None
_initial_col = SELECTED_COLUMN if SELECTED_COLUMN in plottable else _proposed

column_dd = widgets.Dropdown(options=_column_options, value=_initial_col,
                             description="Data column:", style={"description_width": "initial"},
                             layout=widgets.Layout(width="620px"))
confirm_col_btn = widgets.Button(description="Confirm column", button_style="success", icon="check")
confirm_col_out = widgets.Output()

def _on_confirm_col(_):
    global SELECTED_COLUMN
    SELECTED_COLUMN = column_dd.value
    with confirm_col_out:
        confirm_col_out.clear_output()
        print(f"Confirmed column: {_column_label(SELECTED_COLUMN)}")

confirm_col_btn.on_click(_on_confirm_col)
display(widgets.VBox([column_dd, confirm_col_btn, confirm_col_out]))

with confirm_col_out:
    if SELECTED_COLUMN in plottable:
        print(f"Confirmed column: {SELECTED_COLUMN} (already confirmed - you can move on)")
    else:
        print(f"Not confirmed yet - will default to '{_proposed}'.")

### Apply the selection

In [ ]:
SELECTED_COLUMN = globals().get("SELECTED_COLUMN")
if SELECTED_COLUMN not in plottable:
    SELECTED_COLUMN = _proposed
    print(f"No column confirmed - defaulting to '{SELECTED_COLUMN}'.")

DATA_COLUMN = SELECTED_COLUMN

# Axis labels from the metadata, with a plain fallback if the column is not described there
x_vt = column_value_types.get(TIME_COLUMN)
y_vt = column_value_types.get(DATA_COLUMN)
x_axis_label = f"{x_vt.self.label} ({TIME_COLUMN})" if x_vt else TIME_COLUMN
y_axis_label = (f"{y_vt.self.label} [{y_vt.unit}]" if y_vt and getattr(y_vt, "unit", None)
                else (y_vt.self.label if y_vt else DATA_COLUMN))

# Quality flag: gas products have a single 'Flag' column, the meteo product one per
# variable ('AT-Flag'). Resolve it for the selected column, or disable filtering if absent.
if f"{DATA_COLUMN}-Flag" in df.columns:
    QUALITY_FLAG = f"{DATA_COLUMN}-Flag"
elif DEFAULT_QUALITY_FLAG in df.columns:
    QUALITY_FLAG = DEFAULT_QUALITY_FLAG
else:
    QUALITY_FLAG = None

print(f"Plotting column: {DATA_COLUMN}")
print(f"Y axis label:    {y_axis_label}")
print(f"Quality flag:    {QUALITY_FLAG if QUALITY_FLAG else 'none found - no filtering applied'}")

## 8. Make a plot: single data column

Before the data is plotted, the `Flag` column is used to exclude data that has not undergone or passed manual quality control. Note that by default the data-fetching methods of the ICOS Python library already exclude data points explicitly flagged as bad; this is controlled by the `keep_bad_data` parameter of `get_columns_as_arrays` and `batch_get_columns_as_arrays` (`False` by default).

The names of the timestamp and quality-flag columns can differ between data types, depending on the conventions used by the corresponding thematic centre. They are set once in the Configuration cell.

In [ ]:
def assert_cols(frame: pd.DataFrame, cols) -> None:
    """Fail early with a helpful message if an expected column is missing."""
    for col in cols:
        assert col in frame.columns, (
            f"Column '{col}' not found in data. Available columns: {list(frame.columns)}"
        )

assert_cols(df, [TIME_COLUMN, DATA_COLUMN])

if QUALITY_FLAG:
    df_quality = df[df[QUALITY_FLAG] == VALUE_ACCEPT_QUALITY].copy()
else:
    df_quality = df.copy()
df_quality[TIME_COLUMN] = pd.to_datetime(df_quality[TIME_COLUMN], errors="coerce")
df_quality = df_quality.dropna(subset=[TIME_COLUMN, DATA_COLUMN])

print(f"{len(df_quality)} of {len(df)} rows kept"
      + (f" after filtering on {QUALITY_FLAG}." if QUALITY_FLAG else " (no quality filtering)."))

plot = df_quality.plot(x=TIME_COLUMN, y=DATA_COLUMN, grid=True,
                       title=f"{station_name} - {height} m", style="o", markersize=3)
plot.set(xlabel=x_axis_label, ylabel=y_axis_label)
register_figure(plot.get_figure(), "single_series")
plt.show()

### Interactive version of the same plot

In [ ]:
fig = px.scatter(
    df_quality,
    x=TIME_COLUMN,
    y=DATA_COLUMN,
    title=f"{station_name} - {height} m",
    labels={TIME_COLUMN: x_axis_label, DATA_COLUMN: y_axis_label},
)
fig.update_traces(mode="markers")
fig.show()

## 9. Combine all data objects for the station

Each measurement height is a separate time series, so the files are grouped by height and concatenated.

Note that this section downloads one file per day per height, so it can take a while on stations with a long NRT record; `MAX_OBJECTS_TO_COMBINE` in the Configuration cell caps the number of files fetched.

In [ ]:
all_height_dfs = []
n_fetched = 0
n_failed = 0

for h, objects in sorted(file_groups.items()):
    daily_dfs = []
    for dobj in sorted(objects, key=lambda o: o.filename):
        if n_fetched >= MAX_OBJECTS_TO_COMBINE:
            break
        try:
            day_meta = meta.get_dobj_meta(dobj)
            df_day = pd.DataFrame(data.get_columns_as_arrays(day_meta))
            df_day["height"] = h
            df_day["source_file"] = dobj.filename
            df_day[TIME_COLUMN] = pd.to_datetime(df_day[TIME_COLUMN], errors="coerce")
            daily_dfs.append(df_day)
            n_fetched += 1
        except Exception as exc:  # noqa: BLE001 - report and continue with the other files
            n_failed += 1
            print(f"Could not read {dobj.filename}: {type(exc).__name__}: {exc}")

    if daily_dfs:
        all_height_dfs.append(pd.concat(daily_dfs, ignore_index=True).sort_values(TIME_COLUMN))

if not all_height_dfs:
    raise RuntimeError("No data could be read for any height.")

df_all = pd.concat(all_height_dfs, ignore_index=True)
# Unfiltered copy: the rows dropped below are those where the *selected* column is missing,
# which would also remove perfectly good values of the other columns. Sections 10 and 11
# start from this copy instead.
df_all_raw = df_all.copy()
print(f"Fetched {n_fetched} file(s)" + (f", {n_failed} failed" if n_failed else "") + ".")
if n_fetched >= MAX_OBJECTS_TO_COMBINE:
    print(f"NOTE: stopped at MAX_OBJECTS_TO_COMBINE = {MAX_OBJECTS_TO_COMBINE}; "
          f"raise it in the Configuration cell to include more files.")

df_all = df_all.dropna(subset=[TIME_COLUMN, DATA_COLUMN])
if QUALITY_FLAG and QUALITY_FLAG in df_all.columns:
    df_all = df_all[df_all[QUALITY_FLAG] == VALUE_ACCEPT_QUALITY]
df_all = df_all.sort_values(TIME_COLUMN)
print(f"{len(df_all)} rows"
      + (f" after filtering on {QUALITY_FLAG}" if QUALITY_FLAG else " (no quality filtering)")
      + f", heights: {sorted(df_all['height'].unique())}")

### Plot: one time series per measurement height

In [ ]:
fig = plt.figure(figsize=(12, 6))
for h, g in df_all.groupby("height"):
    plt.plot(g[TIME_COLUMN], g[DATA_COLUMN], ".", markersize=3, label=f"{h} m")

plt.grid(True)
plt.xlabel("Time")
plt.ylabel(y_axis_label)
plt.title(f"{station_name} - {DATA_COLUMN} time series by measurement height")
plt.legend()
register_figure(fig, "series_by_height")
plt.show()

### Interactive version

In [ ]:
fig = px.scatter(
    df_all,
    x=TIME_COLUMN,
    y=DATA_COLUMN,
    color="height",
    title=f"{station_name} - {DATA_COLUMN} time series by measurement height",
    labels={TIME_COLUMN: "Time", DATA_COLUMN: y_axis_label, "height": "Height (m)"},
)
fig.update_traces(
    mode="markers",
    hovertemplate=f"Time: %{{x}}<br>{DATA_COLUMN}: %{{y}}<br>Height: %{{marker.color}}<extra></extra>",
)
fig.show()

## 10. Compare several series

The plots above show one variable across measurement heights. This section builds a wide table with
one column per **variable and height**, so that different variables — or the same variable at
different heights — can be compared directly.

It is most useful for products that carry several variables in the same file, such as the meteo
product (`AT`, `RH`, `WS`, ...), and for comparing a gas measured at two heights.

In [ ]:
def _is_ancillary(col):
    """Statistics and flags carry the same generic names in every file, and are not series."""
    return col.endswith(("Stdev", "NbPoints", "Flag"))

wide_value_cols = [c for c in df_all_raw.columns
                   if c not in (TIME_COLUMN, "height", "source_file")
                   and not _is_ancillary(c)
                   and pd.api.types.is_numeric_dtype(df_all_raw[c])]

if not wide_value_cols:
    raise RuntimeError(f"No numeric series found. Columns: {list(df_all_raw.columns)}")

merged_df = pd.DataFrame(columns=[TIME_COLUMN])
series_labels = {}

for h in sorted(df_all_raw["height"].dropna().unique()):
    block = df_all_raw[df_all_raw["height"] == h]
    keep = {}
    for col in wide_value_cols:
        if col not in block.columns or block[col].isna().all():
            continue
        # Quality flag per variable when the product provides one, otherwise the shared column
        flag = f"{col}-Flag" if f"{col}-Flag" in block.columns else (
            DEFAULT_QUALITY_FLAG if DEFAULT_QUALITY_FLAG in block.columns else None)
        series = block[[TIME_COLUMN, col]]
        if flag:
            series = block.loc[block[flag] == VALUE_ACCEPT_QUALITY, [TIME_COLUMN, col]]
        if series[col].dropna().empty:
            continue

        name = f"{col}_{h}"
        n = 2
        while name in merged_df.columns:      # guard against any residual clash
            name = f"{col}_{h}_{n}"
            n += 1
        keep[name] = series.rename(columns={col: name})

        vt = column_value_types.get(col)
        unit = f" [{vt.unit}]" if vt and getattr(vt, "unit", None) else ""
        label = f"{vt.self.label}{unit}" if vt else col
        series_labels[name] = f"{label} at {h} m"

    for name, part in keep.items():
        merged_df = pd.merge(merged_df, part, on=TIME_COLUMN, how="outer")

if merged_df.empty or not series_labels:
    raise RuntimeError("No series could be built for this selection.")

merged_df[TIME_COLUMN] = pd.to_datetime(merged_df[TIME_COLUMN], errors="coerce")
merged_df = merged_df.dropna(subset=[TIME_COLUMN]).sort_values(TIME_COLUMN)

print(f"{len(series_labels)} series over {len(merged_df)} rows:")
for name, lbl in series_labels.items():
    print(f"  {name:28} {lbl}")

### Plot the selected series together

Select the series (ctrl-click / cmd-click for several) and press **Plot**. Variables with very
different value ranges do not sit well on a single axis — use the two-axis plot below for those.

In [ ]:
series_names = list(series_labels)

series_select = widgets.SelectMultiple(
    options=[(f"{n} - {series_labels[n]}", n) for n in series_names],
    value=tuple(series_names[:3]),
    description="Series:", style={"description_width": "initial"},
    rows=min(10, max(3, len(series_names))),
    layout=widgets.Layout(width="680px"))
plot_btn = widgets.Button(description="Plot selected series", button_style="success", icon="check")
plot_out = widgets.Output()

def _plot_series(_=None):
    with plot_out:
        plot_out.clear_output(wait=True)
        cols = list(series_select.value)
        if not cols:
            print("Select at least one series.")
            return
        ax = merged_df.plot(x=TIME_COLUMN, y=cols, grid=True, style=".", markersize=3,
                            figsize=(11, 5), title=station_name)
        ax.legend([series_labels[c] for c in cols], fontsize=8)
        ax.set_xlabel("Time")
        register_figure(ax.get_figure(), "selected_series")
plt.show()

plot_btn.on_click(_plot_series)
display(widgets.VBox([series_select, plot_btn, plot_out]))
_plot_series()

## 11. Two series on separate axes

Useful when the two series have different units or value ranges — for example air temperature and
relative humidity, or the same gas at two measurement heights.

In [ ]:
left_dd = widgets.Dropdown(
    options=[(f"{n} - {series_labels[n]}", n) for n in series_names],
    value=series_names[0], description="Left axis:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="680px"))
right_dd = widgets.Dropdown(
    options=[("(none)", None)] + [(f"{n} - {series_labels[n]}", n) for n in series_names],
    value=series_names[1] if len(series_names) > 1 else None,
    description="Right axis:", style={"description_width": "initial"},
    layout=widgets.Layout(width="680px"))
two_btn = widgets.Button(description="Plot", button_style="success", icon="check")
two_out = widgets.Output()

def _plot_two(_=None):
    with two_out:
        two_out.clear_output(wait=True)
        left, right = left_dd.value, right_dd.value
        fig, ax1 = plt.subplots(figsize=(11, 5))
        ax1.plot(merged_df[TIME_COLUMN], merged_df[left], "b.", markersize=3)
        ax1.set_xlabel("Time")
        ax1.set_ylabel(series_labels[left], color="b")
        ax1.tick_params(axis="y", labelcolor="b")
        ax1.set_title(station_name)

        if right and right != left:
            ax2 = ax1.twinx()
            ax2.plot(merged_df[TIME_COLUMN], merged_df[right], "r.", markersize=3)
            ax2.set_ylabel(series_labels[right], color="r")
            ax2.tick_params(axis="y", labelcolor="r")
        elif right == left:
            print("The two axes show the same series - pick a different one on the right.")

        ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax1.tick_params(axis="x", rotation=45)
        ax1.grid(True)
        fig.tight_layout()
        register_figure(fig, "two_axes")
        plt.show()

two_btn.on_click(_plot_two)
display(widgets.VBox([left_dd, right_dd, two_btn, two_out]))
_plot_two()

<a id="export"></a>
## Take the results with you

The figures above are only displayed in the notebook: nothing has been written to disk while you
worked, and the output folder is emptied every time the notebook is run again.

The cell below writes out the figures that were drawn, packs them into a single zip and gives you a
download link. Run it before starting a new run if you want to keep anything.

The archive is written into an `exports/` folder next to the notebook. That folder is emptied at the start of each run as well, so download what you need before running the notebook again.

In [ ]:
import os
import zipfile

from IPython.display import HTML

RESULT_SUFFIXES = (".png", ".csv", ".txt")


def _human(n):
    for unit in ("B", "kB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024


bundle_btn = widgets.Button(description="Create archive", button_style="success",
                            icon="file-archive-o")
bundle_out = widgets.Output()


def _make_bundle(_=None):
    with bundle_out:
        bundle_out.clear_output()

        # The figures live in memory only: the inline backend closes them at the end of each
        # cell, so they are written out here rather than while plotting.
        written = []
        for stem, fig in DRAWN_FIGURES.items():
            path = OUTPUT_DIR / f"{stem}.png"
            if not path.exists():
                fig.savefig(path, dpi=150, bbox_inches="tight")
                written.append(path.name)
        if written:
            print(f"Wrote {len(written)} figure(s): {', '.join(written)}")

        files = [f for f in sorted(OUTPUT_DIR.rglob("*"))
                 if f.is_file() and f.suffix.lower() in RESULT_SUFFIXES]
        if not files:
            print("Nothing to pack yet: no figure has been drawn in this session. "
                  "Run the plot cells above first.")
            return

        archive = EXPORT_DIR / f"{NOTEBOOK_SLUG}_results.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in files:
                zf.write(f, f.relative_to(OUTPUT_DIR))

        total = sum(f.stat().st_size for f in files)
        print(f"{len(files)} file(s), {_human(total)} -> {_human(archive.stat().st_size)} compressed")
        for f in files:
            print(f"   {f.relative_to(OUTPUT_DIR)}")

        # A plain link would make JupyterLab try to *open* the zip, which fails with "is not
        # UTF-8 encoded". The /files/ endpoint hands it to the browser instead, and under
        # JupyterHub it has to carry the /user/<name>/ prefix or it reaches the hub and
        # returns an HTML error page.
        try:
            rel = archive.resolve().relative_to(Path.home())
        except ValueError:
            rel = archive.name
        _prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/").rstrip("/")
        display(HTML(
            f'<p style="margin-top:8px"><a href="{_prefix}/files/{rel}?download=1" download '
            f'target="_blank" style="padding:8px 14px;background:#1B4053;color:white;'
            f'text-decoration:none;border-radius:5px;display:inline-block">'
            f'Download {archive.name}</a></p>'
            f'<p style="font-size:90%;color:#555">If the link does nothing, right-click '
            f'<code>{archive.name}</code> in the file browser on the left and choose '
            f'<i>Download</i>.</p>'))


bundle_btn.on_click(_make_bundle)
display(widgets.VBox([bundle_btn, bundle_out]))
print("Press 'Create archive' when you want to keep the figures of this run.")